# **6.4 多输入输出通道**

## 6.4.1 从单输入输出到多输入输出通道

在6.1.3当中，我们提到过对通道进行升维。低维转高维，不难理解。我们直接看下图。

![alt text](https://zh.d2l.ai/_images/conv-multi-in.svg)

此为对多通道输入单通道输出。即对不同通道取不同的卷积核，并对卷积的结果进行累加。此时卷积核维度与输入一致。

显然，对卷积核升一个维度便可实现多通道输入输出。

下面是代码实现。

In [1]:
import torch
from d2l import torch as d2l

def corr2d_multi_in(X, K):
    # 先遍历“X”和“K”的第0个维度（通道维度），再把它们加在一起
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))

X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

def corr2d_multi_in_out(X, K):
    # 迭代“K”的第0个维度，每次都对输入“X”执行互相关运算。
    # 最后将所有结果都叠加在一起
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

K = torch.stack((K, K + 1, K + 2), 0)

corr2d_multi_in_out(X, K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

# 6.4.2 $1 \times 1$卷积层

对于单通道输入，$1 \times 1$ 卷积层没有什么用。但在多通道输入中，$1 \times 1$ 唯一的计算发生在通道上，使得卷积层只对图像的通道进行处理，这在特殊场景下会很有用。

![alt text](https://zh-v2.d2l.ai/_images/conv-1x1.svg)

代码实现：

In [ ]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i, h * w))
    #flatten
    K = K.reshape((c_o, c_i))
    # 全连接层中的矩阵乘法
    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))

X = torch.normal(0, 1, (3, 3, 3)) # normal正态分布，输入通道数为3，宽和高均为3
K = torch.normal(0, 1, (2, 3, 1, 1))

Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6
# assert判断和是否小于1e-6，说明两者相等

### **小结**

相当于做了一次维度推广，并阐明了特殊$1 \times 1$形状卷积层的作用